In [ ]:
import glob, json, pathlib
from tqdm.notebook import tqdm

sanitychecks = list(
    glob.glob(r"D:\BEHAVIOR-1K\asset_pipeline\cad\*\*\artifacts\sanitycheck.json")
)
jsons = {}
for x in tqdm(sanitychecks):
    with open(x, "r") as f:
        name = "/".join(pathlib.Path(x).parts[-4:-2])
        jsons[name] = json.load(f)

In [ ]:
total = len(jsons)
failed = {name for name, x in jsons.items() if x["errors"]}
total, len(failed)

In [ ]:
from collections import defaultdict
import re

error_templates_and_ignore = {
    "Category (.*) for object .* does not exist on spreadsheet.": False,
    "(.*) has too many vertices: .* > .*": True,
    "(.*) link .* has different .* in instance .* compared to instance .*. .* difference: .*.": False,
    "(.*) is missing in instance .*, so relative transform check cannot be completed.": False,
    "(.*) has bad name.": False,
    "(.*) has negative object offset scale.*": False,
    "(.*) has scale that is not 1. Reset scale.": False,
    ".*No instance ID 0 instance of (.*)": False,
    "Articulated object (.*) instances have different scales for base links. This may have broken things during the match links script.": True,
    "All instances of (.*) do not have contiguous instance IDs. Missing: .*": False,
    "(.*) has disallowed type.*": False,
    "Light object (.*) should not have object offset rotation. Reset pivot.": False,
    "(.*) has different pivot offset position \(by (.*)\). Match pivots on each instance.": False,
    "(.*) has different pivot offset rotation \(by (.*)\). Match pivots on each instance.": False,
    "(.*) has different shear. Match scaling axes on each instance.": False,
    "Model ID (.*) contains 'todo'.": False,
    "(.*) should not have an upper side.": False,
    "(.*) should have an upper side.": False,
    "(.*) has different vertex count than recorded in provider.*": False,
    "(.*) has different face count than recorded in provider.*": False,
    "(.*) has no collision mesh. Create a collision mesh.": False,
    "Inconsistent link sets within model (.*)": False,
    "(.*) is missing meta link: fillable.": False,
    "(.*) is missing meta link: heatsource.": False,
    "(.*) is missing meta link: togglebutton.": False,
    "(.*) is missing meta link: particleapplier.": False,
    "(.*) is missing meta link: fluidsource.": False,
    "(.*) is missing meta link: fluidsink.": False,
    "(.*) is missing meta link: togglebutton.": False,
    "(.*) element .* has too many vertices .*": True,
    "Model ID (.*) requires joints but has no joints.": False,
    "Expected meta links for (.*) are missing: .*": False,
    "Convex mesh fillable meta link (.*) should be of Editable Poly instead of .*": False,
    "Cloth object (.*) should consist of exactly 1 element. Currently it has .* elements.": False,
    "Cannot validate clothness: category (.*) not found in taxonomy.": False,
    "(.*) element .* is not a volume": True,
    "(.*) element .* has elements trimesh still finds splittable": False,
    "Object (.*) has .* fillable meshes. Should have no more than one.": False,
    "Model (.*) has category .* that is neither the from or to element in the rename file.": False,
    "(.*) has meta link not required by its synset: .*": True,
    "(.*) meta type .* ID .* has non-continuous subids .*": False,
    "Cannot validate meta links and joints for model ID (.*): Category .* not found in taxonomy.": False,
    "(.*) is in the deletion queue. Delete the object.": False,
    "(.*) has unapplied rename .*.": True,
    "fillable_seed is an invalid meta link under (.*).": False,
    "(.*) is not unique.": False,
    "Object (.*) has .* meshes. Should have no more than one.": False,
    "(.*) is not under another object but contains part tags {'connectedpart'}.": False,
    "(.*) is not rendering the baked material. Select the baked material for rendering or rebake.": True,
    "(.*) has non-shell material. Run texture baking.": True,
    "(.*) is not rendering the baked material in the viewport. Select the baked material for viewport.": True,
    "(.*) has different material. Match materials on each instance.": True,
    "(.*) has different UV unwrapping than recorded. Reunwrap the object.": True,
    "Non-top level material .* of (.*) is not a MultiMaterial or some kind of VRay material: .*": True,
    "Collision mesh (.*) has bounding box .* that is more than 5cm different from parent .*": True,
    "(.*) has more than one MultiMaterial in its material hierarchy. This is a bad attachment and has resulted in face material assignment loss.": True,
}

SKIP_IGNORE = False

error_templates = list(error_templates_and_ignore.keys())
error_matches = defaultdict(set)
error_matches_by_file = defaultdict(lambda: defaultdict(set))
unknown_errors_by_file = defaultdict(set)
for name, f in jsons.items():
    for error in f["errors"]:  # + f["warnings"]:
        for i, tmpl in enumerate(error_templates):
            match = re.fullmatch(tmpl, error, re.S)
            if match is not None:
                if not (SKIP_IGNORE and error_templates_and_ignore[tmpl]):
                    error_matches[tmpl].add(match)
                    error_matches_by_file[tmpl][name].add(match)

                break
        else:
            print(error)
            unknown_errors_by_file[name].add(error)

print("\nCounts")
for tmpl, objs in sorted(error_matches.items(), key=lambda x: len(x[1]), reverse=True):
    print(f"{tmpl}: {len(objs)}")

print("\n\nCounts by file")
for tmpl, matches_by_file in error_matches_by_file.items():
    all_matches = error_matches[tmpl]
    if not all_matches:
        continue
    print(f"\n{tmpl}")
    for name, matches in sorted(
        matches_by_file.items(), key=lambda x: x[1], reverse=True
    ):
        if not matches:
            break
        print(f"  {name}: {len(matches)}")

In [ ]:
import sys

sys.path.append(r"D:\BEHAVIOR-1K\asset_pipeline")
from b1k_pipeline.utils import parse_name

error_objs = {
    parse_name(x.group(1)).group("model_id")
    for x in error_matches[
        "(.*) material tree contains .* with disallowed type .*. It should be one of .*."
    ]
}
print(len(error_objs))
# with open("bad_materials.json", "w") as f:
#     json.dump(sorted(error_objs), f)

In [ ]:
from IPython.display import display, Markdown

# Print the number of kinds of error for each target
output = []
output.append("\n\n# Counts by error")
for name in jsons.keys():
    all_errors = {}
    for tmpl, matches_by_file in error_matches_by_file.items():
        if error_templates_and_ignore[tmpl]:
            continue
        matches = matches_by_file[name]
        if not matches:
            continue
        if (
            tmpl == "(.*) is in the deletion queue. Delete the object."
            and "objects/" in name
        ):
            continue
        all_errors[tmpl] = matches

    for error in unknown_errors_by_file[name]:
        all_errors["Unknown"] = all_errors.get("Unknown", set())
        all_errors["Unknown"].add(error)

    if all_errors:
        output.append(f"\n## {name}")
        for k, v in sorted(all_errors.items(), key=lambda x: len(x[1]), reverse=True):
            output.append(f"  ### {k}: {len(v)}")
            for match in v:
                err_str = match if isinstance(match, str) else match.group(0)
                output.append("    " + err_str)
            output.append("")

display(Markdown("\n".join(output)))

In [ ]:
from collections import defaultdict
import re

warning_re = re.compile(r"(.*) has too many vertices: (\d+) > 20000")
warn_objs = defaultdict(dict)
for name, x in jsons.items():
    for w in x["warnings"]:
        m = warning_re.fullmatch(w)
        if m:
            warn_objs[name][m.group(1)] = int(m.group(2))

print(warn_objs)

In [ ]:
# Worst offenders
sorted(
    (cnt, obj) for warn_target in warn_objs.values() for obj, cnt in warn_target.items()
)

In [ ]:
# Check which of the "articulated object has diff scale" errors are on bad objects
providers = json.loads(
    pathlib.Path(
        r"D:\BEHAVIOR-1K\asset_pipeline\artifacts\pipeline\object_inventory.json"
    ).read_text()
)["providers"]
id_providers = {k.split("-")[-1]: v for k, v in providers.items()}
categories = dict([tuple(reversed(k.split("-"))) for k in providers.keys()])
articulation_tmpl = "Articulated object (.*) instances have different scales for base links. This may have broken things during the match links script."
for target, objs in error_matches_by_file[articulation_tmpl].items():
    if not objs:
        continue
    bad_objs = set()
    for match in objs:
        obj = match.group(1)
        if id_providers[obj] != target:
            name = f"{categories[obj]}-{obj}"
            bad_objs.add(name)

    if not bad_objs:
        continue

    print(f"{target}: {len(bad_objs)} {bad_objs}")

In [ ]:
# Get a list of all the objects that show up in the match links relevant errors
mismatched_link_offset_tmpl = "(.*) link .* has different .* in instance .* compared to instance .*. .* difference: .*."
objects_for_match_links = (
    error_matches[mismatched_link_offset_tmpl] | error_matches[articulation_tmpl]
)
print(
    ", ".join(
        f'"{x.group(1)}"'
        for x in sorted(objects_for_match_links, key=lambda x: x.group(1))
    )
)

In [ ]:
import sys

sys.path.append(r"D:\BEHAVIOR-1K\asset_pipeline")
from b1k_pipeline.utils import parse_name

# Get a link of all the objects that are bad cloth
cloth_tmpl = "Cloth object (.*) should consist of exactly 1 element. Currently it has .* elements."
objects_for_cloth = [
    parse_name(x.group(1)).group("model_id") for x in error_matches[cloth_tmpl]
]
print(", ".join(f'"{x}"' for x in sorted(objects_for_cloth)))